# Task 3: Bring Your Own Dataset - Diamonds Data Analysis

This notebook explores the **Diamonds** dataset, which contains the prices and other attributes of almost 54,000 diamonds.
The goal is to demonstrate loading CSV data, performing basic exploration, filtering, and aggregation using PySpark, and finally storing the result in Delta Lake format on MinIO.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count
import pandas as pd
import os

# Initialize Spark Session with local mode for reliability
spark = SparkSession.builder \
    .appName("DiamondsAnalysis") \
    .master("local[*]") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("SparkSession created.")

# 3.1 Download the dataset
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
diamonds_pd = pd.read_csv(url)
diamonds_pd.to_csv("/home/jovyan/work/diamonds.csv", index=False)

print(f"Dataset downloaded: {len(diamonds_pd)} rows.")

SparkSession created.
Dataset downloaded: 53940 rows.


In [3]:
# a) Load the CSV and print the schema
df = spark.read.csv("/home/jovyan/work/diamonds.csv", header=True, inferSchema=True)
print("=== Schema ===")
df.printSchema()

=== Schema ===
root
 |-- carat: double (nullable = true)
 |-- cut: string (nullable = true)
 |-- color: string (nullable = true)
 |-- clarity: string (nullable = true)
 |-- depth: double (nullable = true)
 |-- table: double (nullable = true)
 |-- price: integer (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- z: double (nullable = true)



In [4]:
# b) Use df.describe().show() to see basic statistics
print("=== Basic Statistics ===")
df.describe().show()

=== Basic Statistics ===
+-------+------------------+---------+-----+-------+------------------+------------------+-----------------+------------------+------------------+------------------+
|summary|             carat|      cut|color|clarity|             depth|             table|            price|                 x|                 y|                 z|
+-------+------------------+---------+-----+-------+------------------+------------------+-----------------+------------------+------------------+------------------+
|  count|             53940|    53940|53940|  53940|             53940|             53940|            53940|             53940|             53940|             53940|
|   mean|0.7979397478679852|     NULL| NULL|   NULL| 61.74940489432624| 57.45718390804603|3932.799721913237| 5.731157211716609| 5.734525954764462|3.5387337782723316|
| stddev|0.4740112444054196|     NULL| NULL|   NULL|1.4326213188336525|2.2344905628213247|3989.439738146397|1.1217607467924915|1.1421346741235616

In [5]:
# c) Filter rows by a meaningful condition (Premium diamonds over 1 carat)
print("=== Filtering: Premium cut and carat > 1.0 ===")
filtered_df = df.filter((col("cut") == "Premium") & (col("carat") > 1.0))
print(f"Filtered count: {filtered_df.count()}")
filtered_df.show(5)

=== Filtering: Premium cut and carat > 1.0 ===
Filtered count: 5729
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
|carat|    cut|color|clarity|depth|table|price|   x|   y|   z|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
| 1.01|Premium|    F|     I1| 61.8| 60.0| 2781|6.39|6.36|3.94|
| 1.01|Premium|    H|    SI2| 62.7| 59.0| 2788|6.31|6.22|3.93|
| 1.04|Premium|    G|     I1| 62.2| 58.0| 2801|6.46|6.41| 4.0|
| 1.02|Premium|    G|     I1| 60.3| 58.0| 2815|6.55| 6.5|3.94|
| 1.17|Premium|    J|     I1| 60.2| 61.0| 2825| 6.9|6.83|4.13|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
only showing top 5 rows



In [6]:
# d) Group by a categorical column (cut) and compute an aggregate (avg price)
print("=== Aggregation: Average Price by Cut ===")
agg_df = df.groupBy("cut").agg(
    avg("price").alias("avg_price"),
    count("*").alias("count")
).orderBy(col("avg_price").desc())

agg_df.show()

=== Aggregation: Average Price by Cut ===
+---------+------------------+-----+
|      cut|         avg_price|count|
+---------+------------------+-----+
|  Premium|4584.2577042999055|13791|
|     Fair| 4358.757763975155| 1610|
|Very Good|3981.7598907465654|12082|
|     Good| 3928.864451691806| 4906|
|    Ideal| 3457.541970210199|21551|
+---------+------------------+-----+



In [7]:
# e) Write the processed result to Delta Lake format on MinIO
delta_path = "s3a://warehouse/lab01/my_dataset"
print(f"Writing to Delta Lake: {delta_path}...")
agg_df.write.format("delta").mode("overwrite").save(delta_path)
print("Write completed.")

Writing to Delta Lake: s3a://warehouse/lab01/my_dataset...
Write completed.


In [8]:
# f) Read it back from Delta Lake and verify the row count matches
print("=== Verification: Reading back from Delta Lake ===")
read_back_df = spark.read.format("delta").load(delta_path)
print(f"Original Aggregated Row Count: {agg_df.count()}")
print(f"Read Back Row Count: {read_back_df.count()}")

if agg_df.count() == read_back_df.count():
    print("✅ Success! Row counts match.")
else:
    print("❌ Failure! Row counts do not match.")

read_back_df.show()

=== Verification: Reading back from Delta Lake ===
Original Aggregated Row Count: 5
Read Back Row Count: 5
✅ Success! Row counts match.
+---------+------------------+-----+
|      cut|         avg_price|count|
+---------+------------------+-----+
|  Premium|4584.2577042999055|13791|
|     Fair| 4358.757763975155| 1610|
|Very Good|3981.7598907465654|12082|
|     Good| 3928.864451691806| 4906|
|    Ideal| 3457.541970210199|21551|
+---------+------------------+-----+



Read Back Row Count: 5


✅ Success! Row counts match.


+---------+------------------+-----+
|      cut|         avg_price|count|
+---------+------------------+-----+
|  Premium|4584.2577042999055|13791|
|     Fair| 4358.757763975155| 1610|
|Very Good|3981.7598907465654|12082|
|     Good| 3928.864451691806| 4906|
|    Ideal| 3457.541970210199|21551|
+---------+------------------+-----+



In [9]:
spark.stop()